# 07 Data Leakage Audit

**Author:** Rowan Walker

This notebook presents a thorough investigation into the model to identify if there are any instances of data leakage. This is a critical and often hidden weakness in prediction models; using data for in training that would not be available at the time of inference. This can massively inflate back testing results, hence it is incumbent on a developer of a model to remove any potential source of data leakage.

### 7.1 Source 1: Feature creation

The first potential source of data leakage is in feature creation. Below we investigate the model code to ensure that it is leakage proof.

In [ ]:
def create_features(
    ticker: str,
    hold_days: int,
) -> pd.DataFrame:
    """
    Build the full feature set for a single ticker by combining its price series
    with macro inputs and a range of technical indicators.

    The function:
         - Loads the ticker's price history
         - Merges external macro series (rf, FTSE index, FX, commodities)
         - Computes rolling statistical features (returns, volatility, beta)
         - Builds technical indicators (RSI, moving averages, correlations)
         - Creates the binary label based on forward returns

    Parameters
    ----------
    ticker : str
        The equity ticker to process.
    hold_days : int
        Forward return horizon used to generate the classification label.

    Returns
    -------
    pd.DataFrame
        Feature matrix with predictors and the final label column.
    """
    raw_paths = data_cfg["paths"]["raw"]

    # FTSE index benchmark
    index_file = "index.parquet"
    ftse = (
        pd.read_parquet(raw_paths["base"] / index_file)
        .rename(columns={"Close": "ftse"})
        .set_index("Date")
    )
    ftse.index = pd.to_datetime(ftse.index)

    # Load raw price data for the ticker
    df = pd.read_parquet(raw_paths["ftse"] / f"{ticker}.parquet")
    df.index = pd.to_datetime(df.index)

    # Short, medium, long-term windows
    sml = (5, 21, 60)

    df = df.join(ftse[["ftse"]], how="left")

    # Forward returns and label
    df[f"Return_{hold_days}d"] = (
        df["Close"].pct_change(hold_days).shift(-hold_days)
    )
    df["Label"] = (df[f"Return_{hold_days}d"] > 0).astype(int)

    # Log returns and smoothed returns
    df["Log_return"] = np.log(df["Close"] / df["Close"].shift(1))
    for w in sml:
        df[f"Log_return_{w}"] = df["Log_return"].ewm(
            span=w,
            adjust=False,
        ).mean()

    # Price-based moving average features
    for w in sml:
        df[f"Price_{w}"] = (
            df["Close"].rolling(w).mean() / df["Close"] - 1
        )

    # Volume moving averages
    for w in sml:
        df[f"Volume{w}"] = df["Volume"].rolling(w).mean()

    # Price-volume correlation
    df["Vol_price_correlation"] = (
        df["Close"].rolling(21).corr(df["Volume"])
    )

    # RSI (14-day)
    delta = df["Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    rs = gain.rolling(14).mean() / loss.rolling(14).mean()
    df["RSI_14"] = 100 - (100 / (1 + rs))

    # Rolling volatility estimates
    for w in sml:
        df[f"Vol_{w}"] = df["Log_return"].rolling(w).std()

    # Temporal features
    df["Day_of_week"] = df.index.dayofweek
    df["Month_of_year"] = df.index.month

    # Benchmark returns and correlation
    df["Log_return_benchmark"] = np.log(
        df["ftse"] / df["ftse"].shift(1)
    )
    df["Benchmark_correlation"] = (
        df["Log_return"]
        .rolling(21)
        .corr(df["Log_return_benchmark"])
    )

    # Rolling beta vs benchmark
    for w in sml:
        rolling_cov = (
            df["Log_return"]
            .rolling(w)
            .cov(df["Log_return_benchmark"])
        )
        rolling_var = (
            df["Log_return_benchmark"]
            .rolling(w)
            .var()
        )
        df[f"Beta_{w}"] = rolling_cov / rolling_var

    # Remove unused columns
    df.dropna(inplace=True)
    df.drop(
        columns=[
            "High",
            "Low",
            "Open",
            "Log_return",
            "Log_return_benchmark",
            f"Return_{hold_days}d",
        ],
        inplace=True,
    )

    # Final feature list
    features = [col for col in df.columns if col != "Label"]

    return df[features + ["Label"]]

#### Summary:
`create_features()` is **safe** from direct leakage as written because:
- Label is forward-looking
- Rolling/EWM features are backward-looking
- Benchmark features are computed using lagged/rolling operations

### 7.2 Source 2: HMM feature creation

In [ ]:
def hmm_features() -> pd.DataFrame:
    """
    Generate technical features for Hidden Markov Model (HMM) regime analysis.

    Features include:
        - Exponentially weighted log returns over multiple windows
        - Price and volume moving averages
        - Price-volume correlation
        - RSI (14-day)
        - Rolling volatility

    Returns
    -------
    pd.DataFrame
        DataFrame containing the engineered features, indexed by Date.
    """
    path = data_cfg["paths"]["raw"]["base"]
    file = "index.parquet"

    # Load price data
    df = pd.read_parquet(path / file).set_index("Date")
    df.index = pd.to_datetime(df.index)

    # Define short, medium, long-term windows
    sml = (5, 21, 60)

    # Log returns and smoothed versions
    df["Log_return"] = np.log(df["Close"] / df["Close"].shift(1))
    for w in sml:
        df[f"Log_return_{w}"] = df["Log_return"].ewm(
            span=w,
            adjust=False,
        ).mean()

    # Price-based moving averages
    for w in sml:
        df[f"Price_{w}"] = df["Close"].rolling(w).mean() / df["Close"] - 1

    # Volume moving averages
    for w in sml:
        df[f"Volume{w}"] = df["Volume"].rolling(w).mean()

    # Price-volume correlation (21-day rolling)
    df["Vol_price_correlation"] = df["Close"].rolling(21).corr(df["Volume"])

    # RSI (14-day)
    delta = df["Close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    rs = gain.rolling(14).mean() / loss.rolling(14).mean()
    df["RSI_14"] = 100 - (100 / (1 + rs))

    # Rolling volatility
    for w in sml:
        df[f"Vol_{w}"] = df["Log_return"].rolling(w).std()

    # Drop raw columns to keep only engineered features
    df.dropna(inplace=True)
    df.drop(
        columns=[
            "Close",
            "High",
            "Low",
            "Open",
            "Volume",
            "Log_return",
        ],
        inplace=True,
    )

    return df


def hmm_model(
    df: pd.DataFrame,
    n_regimes: int,
) -> pd.DataFrame:
    """
    Fit a Gaussian Hidden Markov Model (HMM) to the input features and
    return the posterior probabilities of each regime.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame of engineered features (e.g., technical indicators), indexed by Date.
    n_regimes : int
        Number of latent regimes (HMM components) to fit.

    Returns
    -------
    pd.DataFrame
        DataFrame of shape (len(df), n_regimes) containing the probability
        of each regime at each time step.
    """
    # Convert to float32 numpy array
    df_hmm = df.values.astype(np.float32)

    # Standardise features
    mean = df_hmm.mean(axis=0)
    std = np.clip(df_hmm.std(axis=0), 1e-5, None)
    df_hmm_norm = (df_hmm - mean) / std

    # Fit Gaussian HMM
    model = hmm.GaussianHMM(
        n_components=n_regimes,
        covariance_type="full",
        n_iter=10000,
        tol=1e-4,
        random_state=common_cfg.random_seed,
    )

    model.fit(df_hmm_norm)

    # Compute probabilities for each regime
    states = model.predict_proba(df_hmm_norm)

    # Construct DataFrame with regime probability columns
    regime_cols = [f"Regime_{i}_prob" for i in range(n_regimes)]
    regime_df = pd.DataFrame(states, columns=regime_cols, index=df.index)

    return regime_df

#### Summary
The feature engineering in `hmm_features()` is time-safe.  
However, `hmm_model()` must be fit and normalised **only on the training window**, then applied forward otherwise regime probabilities may leak future information into the training set.

### 7.3 Source 3: Model Training

In [ ]:
def train_model(
    X: np.ndarray,
    y: np.ndarray,
    stock_ids: np.ndarray,
    regime_X: np.ndarray,
    hold_days: int,
    batch_size: int = train_cfg.batch_size,
    weight_decay: float = train_cfg.weight_decay,
    lr: float = train_cfg.lr,
    epochs: int = train_cfg.epochs,
    device: Optional[str] = device,
    model_save_path: str = train_cfg.model_save_path,
    ablation_name: str = 'baseline',
    verbose: bool = True
) -> None:
    """
    Train a TimeSeriesTransformer model on multivariate time series data.

    Parameters
    ----------
    X : np.ndarray
        Input sequences of shape (num_samples, seq_len, feature_dim)
    y : np.ndarray
        Labels of shape (num_samples,)
    stock_ids : np.ndarray
        Stock indices of shape (num_samples,)
    regime_X : np.ndarray
        Regime sequences of shape (num_samples, seq_len, n_regimes)
    hold_days : int
        Forward horizon for the model (used in saving the model)
    batch_size : int, default=BATCH_SIZE
        Batch size for training
    weight_decay : float, default=WEIGHT_DECAY
        Weight decay for AdamW optimizer
    lr : float, default=LR
        Learning rate for optimizer
    epochs : int, default=EPOCHS
        Number of training epochs
    device : str, optional
        Device to train on
    path : str, default='../results/model/'
        Directory to save the trained model
    ablation_name: str, default='baseline'
        Used in ablation testing
    verbose : bool, default=True
        Whether to print training progress
    """
    # Convert data to torch tensors
    X_tensor = torch.tensor(X, dtype=torch.float32)
    y_tensor = torch.tensor(y, dtype=torch.long)
    stock_tensor = torch.tensor(stock_ids, dtype=torch.long)
    regime_tensor = torch.tensor(regime_X, dtype=torch.float32)

    # Create dataset and loader
    dataset = TensorDataset(X_tensor, stock_tensor, regime_tensor, y_tensor)
    
    g = torch.Generator()
    g.manual_seed(common_cfg.random_seed)
    
    loader = DataLoader(
        dataset,
        shuffle=False,
        batch_size=batch_size,
        worker_init_fn=seed_worker,
        generator=g,
        num_workers=os.cpu_count()
    )

    # Initialize model, optimizer, and loss function
    model = TimeSeriesTransformer(
        feature_dim=X.shape[2],
        ablation_name=ablation_name
    ).to(device)

    # Optimiser with weight decay
    decay, no_decay = [], []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.endswith("bias") or "norm" in name.lower():
            no_decay.append(param)
        else:
            decay.append(param)
    
    optimizer = torch.optim.AdamW(
        [
            {"params": decay, "weight_decay": weight_decay},
            {"params": no_decay, "weight_decay": 0.0},
        ],
        lr=lr
    )

    # Learning Rate Scheduler
    total_steps = epochs * len(loader)
    warmup_steps = int(0.1 * total_steps)
    
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return 0.5 * (1 + math.cos(math.pi * progress))
    
    scheduler = LambdaLR(optimizer, lr_lambda)
    loss_fn = nn.CrossEntropyLoss()

    if verbose:
        print(f'Beginning training for {epochs} epochs on device "{device}"')
        print('~' * 60)

    # Training loop
    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        model.train()
        total_loss = 0

        for xb, stock_id, regime, yb in loader:
            xb, stock_id, regime, yb = xb.to(device), stock_id.to(device), regime.to(device), yb.to(device)

            optimizer.zero_grad()
            preds = model(xb, stock_id, regime)
            loss = loss_fn(preds, yb)
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()

        end = time.perf_counter()
        if verbose:
            avg_loss = total_loss / len(loader)
            print(f"Epoch {epoch}: Loss = {avg_loss:.4f}. Time taken = {end - start:.2f}s")

    # Save trained model
    save_path = Path(PROJECT_ROOT / model_save_path)
    save_path.mkdir(parents=True, exist_ok=True)
    file_name = f'model_{hold_days}.pth'
    torch.save(model.state_dict(), save_path / file_name)
    if verbose:
        print('~' * 60)
        print(f'Training completed. Model saved: {save_path / file_name}')

#### Summary:

The `train_model()` function itself is **not directly leaking labels** into the model.  
The forward pass only uses `(X, stock_id, regime_X)` and the loss is computed correctly from `y`.

- No target values are used as inputs to the model
- No evaluation/test information is referenced inside training
- Model saving does not introduce leakage

### 7.4 Source 4: Back testing

In [ ]:
def add_signal(
    long_threshold: float,
    short_threshold: float,
    horizons: Sequence[int] = (1, 5, 21),
    challenger: bool = False,
    df: pd.DataFrame | None = None
) -> pd.DataFrame:
    """
    Generate trading signals based on ensemble ranking of model predictions.

    Parameters
    ----------
    long_threshold : float
        Threshold above which a long position (1) is taken.
    short_threshold : float
        Threshold below which a short position (-1) is taken.
    horizons : Sequence[int], optional
        Prediction horizons to include in ensemble ranking, by default (1,5,21)
    challenger : bool, optional
        If True, uses challenger model predictions, by default False
    df : pd.DataFrame, optional
        If provided, uses this DataFrame instead of reading from file

    Returns
    -------
    pd.DataFrame
        DataFrame with added 'EnsembleRank' and 'Position' columns
    """
    if df is None:
        input_path = data_cfg['paths']['inference']
        input_file = 'inference_challenger.parquet' if challenger else 'inference.parquet'
        df = pd.read_parquet(input_path / input_file)

    rank_cols = []
    for h in horizons:
        col = f"Prediction_{h}"
        if col not in df.columns:
            continue
        rank_col = f"Rank_{h}"
        df[rank_col] = df.groupby("Date")[col].rank(pct=True)
        rank_cols.append(rank_col)

    if not rank_cols:
        raise ValueError(
            f"No prediction columns found. Expected one of: "
            f"{[f'Prediction_{h}' for h in horizons]}"
        )

    df["EnsembleRank"] = df[rank_cols].mean(axis=1)
    df["Position"] = 0
    df.loc[df["EnsembleRank"] >= long_threshold, "Position"] = 1
    df.loc[df["EnsembleRank"] <= short_threshold, "Position"] = -1

    return df


def create_backtest_data(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Prepare price and signal matrices for backtesting.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing at least 'Date', 'Ticker', 'Close', and 'Position'

    Returns
    -------
    Tuple[pd.DataFrame, pd.DataFrame]
        prices : pivoted DataFrame of Close prices (index=Date, columns=Ticker)
        signals : pivoted DataFrame of Position signals (index=Date, columns=Ticker)
                  missing values filled with 0 and cast to int
    """
    prices = pd.pivot_table(df, values='Close', columns='Ticker', index='Date')
    signals = pd.pivot_table(df, values='Position', columns='Ticker', index='Date') \
                .fillna(0).astype(int)
    return prices, signals


def vol_target_weights(
    signals: pd.Series,
    returns: pd.DataFrame,
    target_vol: float,
    max_leverage: float,
    lookback: int = 21,
    ann_factor: int = 252
) -> pd.Series:
    """
    Compute volatility-targeted portfolio weights based on signals.

    Parameters
    ----------
    signals : pd.Series
        Trading signals (+1, 0, -1) for each ticker
    returns : pd.DataFrame
        Daily returns for each ticker, indexed by date
    target_vol : float
        Annualized target portfolio volatility
    max_leverage : float
        Maximum allowed sum of absolute weights
    lookback : int, optional
        Rolling window to compute volatility and covariance, by default 21
    ann_factor : int, optional
        Annualization factor for volatility, by default 252

    Returns
    -------
    pd.Series
        Scaled weights for each ticker, volatility-targeted and leverage-limited
    """
    tickers = signals.index
    vol = returns[tickers].rolling(lookback).std().iloc[-1]
    vol = vol.replace(0, np.nan).fillna(vol.median())
    raw = signals / vol

    if raw.abs().sum() == 0:
        return pd.Series(0, index=tickers)

    weights = raw / raw.abs().sum()
    cov = returns.iloc[-lookback:].cov() * ann_factor
    curr_vol = np.sqrt(weights @ cov @ weights)

    if curr_vol == 0:
        return pd.Series(0, index=tickers)

    scaled = weights * (target_vol / curr_vol)
    total_leverage = scaled.abs().sum()
    if total_leverage > max_leverage:
        scaled *= (max_leverage / total_leverage)

    return scaled


def backtest(
    param_grid: Dict[str, float],
    start_date: str,
    end_date: str,
    df: pd.DataFrame | None = None,
    lookback: int = 21,
    initial_equity: float = 1000,
    sharpe_only: bool = False,
    output: bool = True,
    optimiser: bool = False,
    challenger: bool = False
) -> Union[float, Tuple[pd.Series, pd.Series, pd.Series], Tuple[pd.Series, float]]:
    """
    Backtest a signal-based strategy with risk management, take-profit, stop-loss, and
    volatility-targeted position sizing.

    Parameters
    ----------
    param_grid : dict
        Strategy parameters: long/short thresholds, vol target, slippage, commission, take-profit, stop-loss, max_hold_days, max_drawdown, leverage
    start_date : str
        Backtest start date (YYYY-MM-DD)
    end_date : str
        Backtest end date (YYYY-MM-DD)
    df : pd.DataFrame, optional
        DataFrame to use instead of reading from file, by default None
    lookback : int, optional
        Rolling window for volatility calculation, by default 21
    initial_equity : float, optional
        Starting capital, by default 1000
    sharpe_only : bool, optional
        If True, returns only the Sharpe ratio, by default False
    output : bool, optional
        If True, prints results and plots equity curve, by default True
    optimiser : bool, optional
        If True, returns daily returns and Sharpe ratio for optimization, by default False
    challenger : bool, optional
        If True, uses the challenger model inference, by default False

    Returns
    -------
    float or tuple
        Depending on flags:
        - Sharpe ratio (if sharpe_only=True)
        - Tuple(strategy_equity, benchmark_equity, risk_free) (if sharpe_only=False and optimiser=False)
        - Tuple(daily_returns, sharpe) (if optimiser=True)
    """
    long_threshold = param_grid['long_threshold']
    short_threshold = param_grid['short_threshold']
    target_vol = param_grid['target_vol']
    slippage_bps = param_grid['slippage']
    commission_bps = param_grid['commission']
    take_profit = param_grid['take_profit']
    stop_loss = param_grid['stop_loss']
    max_hold_days = param_grid['max_hold_days']
    max_drawdown = param_grid['max_drawdown']
    fraction_per_trade = param_grid['leverage']

    if df is not None:
        df = df.copy()
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
            df.set_index('Date', inplace=True)

    if df is None:
        df = add_signal(long_threshold=long_threshold,
                        short_threshold=short_threshold,
                        challenger=challenger)
    elif "Position" not in df.columns:
        df = add_signal(long_threshold=long_threshold,
                        short_threshold=short_threshold,
                        df=df)

    df = df.loc[start_date:end_date]
    prices, signals = create_backtest_data(df)
    returns = prices.pct_change(fill_method=None).fillna(0)

    equity_curve = [initial_equity]
    equity = initial_equity
    positions = {t: 0 for t in prices.columns}
    entry_price = {t: None for t in prices.columns}
    holding_days = {t: 0 for t in prices.columns}
    dd_peak = equity

    start_idx = min(lookback + 1, len(prices) - 1)
    for i in range(start_idx, len(prices)):
        date = prices.index[i]
        dd = equity / dd_peak - 1
        if dd < -max_drawdown:
            positions = {t: 0 for t in positions}
            entry_price = {t: None for t in entry_price}
            holding_days = {t: 0 for t in holding_days}
        dd_peak = max(dd_peak, equity)

        todays_signals = signals.loc[date]
        window_returns = returns.iloc[:i]
        target_weights = vol_target_weights(
            todays_signals,
            window_returns,
            target_vol=target_vol,
            lookback=lookback,
            ann_factor=252,
            max_leverage=fraction_per_trade
        )

        for t in positions:
            prev_weight = positions[t]
            new_weight = target_weights[t]
            if prev_weight != new_weight:
                cost = equity * abs(new_weight - prev_weight) * (slippage_bps + commission_bps) / 10000
                equity -= cost
            positions[t] = new_weight
            if new_weight != 0 and entry_price[t] is None:
                entry_price[t] = prices.loc[date, t]
                holding_days[t] = 0

        for t in positions:
            if positions[t] != 0:
                holding_days[t] += 1

        for t in positions:
            if positions[t] == 0 or entry_price[t] is None:
                continue
            current_price = prices.loc[date, t]
            pnl_return = (current_price - entry_price[t]) / entry_price[t] * np.sign(positions[t])
            if pnl_return >= take_profit or pnl_return <= stop_loss or holding_days[t] >= max_hold_days:
                positions[t] = 0
                entry_price[t] = None
                holding_days[t] = 0

        daily_ret = sum(positions[t] * returns.loc[date, t] for t in positions)
        equity *= (1 + daily_ret)
        equity_curve.append(equity)

    curve = pd.Series(
        equity_curve,
        index=prices.index[start_idx-1:]
    )
    
    equity_df = curve.to_frame('Strategy_Equity')

    benchmark_path = data_cfg['paths']['raw']['base']
    benchmark_file = 'index.parquet'
    benchmark = pd.read_parquet(benchmark_path / benchmark_file).set_index('Date')[['Close']]
    benchmark.columns = ['Benchmark_Close']
    benchmark.index = pd.to_datetime(benchmark.index)

    rf_path = data_cfg['paths']['raw']['rf']
    rf_file = 'rf.parquet'
    rf = pd.read_parquet(rf_path / rf_file).set_index('Date').rename(columns={'Close': 'rf'})
    rf.index = pd.to_datetime(rf.index)

    equity_df = equity_df.join(benchmark, how='left').join(rf, how='left')
    equity_df["Benchmark_Equity"] = (1 + equity_df["Benchmark_Close"].pct_change(fill_method=None)).cumprod() * initial_equity

    strategy_equity = equity_df['Strategy_Equity']
    benchmark_equity = equity_df['Benchmark_Equity']
    rf_series = equity_df['rf']
    strategy_returns = compute_returns(strategy_equity)

    if optimiser:
        return strategy_returns, sharpe_ratio(strategy_returns, rf_series)
    if sharpe_only:
        return sharpe_ratio(strategy_returns, rf_series)

    if output:
        plt.style.use('ggplot')
        print("Strategy final equity:", f'{strategy_equity.iloc[-1]:,.0f}')
        print("Buy-and-hold final equity:", f'{benchmark_equity.iloc[-1]:,.0f}')
        print('\n')
        print('Sharpe Ratio:', f'{sharpe_ratio(strategy_returns, rf_series):,.2f}')
        alpha, beta = compute_alpha_beta(strategy_equity, benchmark_equity, rf_series)
        print('Alpha (annualised):', f'{alpha:,.2f}%')
        print('Beta:', f'{beta:,.2f}')
        equity_df[['Strategy_Equity', 'Benchmark_Equity']].plot(figsize=(12, 6))
        plt.title("Strategy vs FTSE 100")
        plt.xlabel("Date")
        plt.ylabel("Equity")
        plt.grid(True)
        plt.show()

    return strategy_equity, benchmark_equity, rf_series

#### Summary

- **Signal Generation (`add_signal`)**  
  - Signals are computed per date based on ensemble rankings of model predictions.  
  - Rankings are grouped by date; future information is never accessed.  
  - Conclusion: No leakage detected.

- **Backtesting (`backtest`)**  
  - Equity and positions are updated chronologically, using only past and current data.  
  - Volatility targeting uses rolling windows of past returns.  
  - Stop-loss, take-profit and maximum holding rules use current or historical prices only.  
  - Benchmark and risk-free returns are merged post hoc and do not influence signals.  
  - Conclusion: No leakage detected.

- **Volatility Targeting (`vol_target_weights`)**  
  - Rolling window calculations are strictly based on past returns.  
  - Leverage scaling does not access future data.  
  - Conclusion: No leakage detected.

### 7.5 Source 5: Pytest: Simple tests for leakage

#### Summary

All tests were run using the `leakage` pytest marker.

1. **Trivial Baseline Sanity Check**  
   - **Test:** `test_no_leakage_baseline_not_perfect`  
   - **Purpose:** Ensures that a simple threshold model using a single feature cannot achieve artificially high out-of-sample accuracy.  
   - **Result:** Passed – Accuracy remained below the suspicious threshold (0.85).

2. **Permutation Test**  
   - **Test:** `test_permutation_test_accuracy_collapses`  
   - **Purpose:** Shuffle training labels and check that a simple rule does not perform well on the test set.  
   - **Result:** Passed – Accuracy dropped to near-random (~0.5), indicating no leakage from features to target.

3. **Forward-Shift Feature Test**  
   - **Test:** `test_shift_features_forward_should_hurt`  
   - **Purpose:** Artificially shifts test features forward to simulate access to future information. Performance should not improve materially.  
   - **Result:** Passed – Out-of-sample accuracy did not improve significantly, confirming features are free from future leaks.

**How to run Leakage Tests**

```bash
pytest -m leakage